<a href="https://colab.research.google.com/github/hinamehboobcs/Industry-Project/blob/main/RuleBasedScheduling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#test data loader
import sys
sys.path.append("/content/RB_Project/modules")

from data_loader import load_carers, load_visits

carers = load_carers("/content/RB_Project/data/carers_data.csv")
visits = load_visits("/content/RB_Project/data/visits_data.csv")

print(carers.head())
print(visits.head())

[DATA LOADER] Carers loaded: (263, 12)
[DATA LOADER] Visits loaded: (657, 9)
  CarerID   Latitude  Longitude  MaxTravelDistanceMiles  \
0   C0001  51.372966  -0.357305                       5   
1   C0001  51.372966  -0.357305                       5   
2   C0001  51.372966  -0.357305                       5   
3   C0001  51.372966  -0.357305                       5   
4   C0002  51.310247  -0.362365                       8   

                                         WorkingDays   OffDays ShiftStart  \
0  Monday, Tuesday, Wednesday, Thursday, Friday, ...    Sunday      07:00   
1  Monday, Tuesday, Wednesday, Thursday, Friday, ...    Sunday      07:00   
2  Monday, Tuesday, Wednesday, Thursday, Friday, ...    Sunday      07:00   
3  Monday, Tuesday, Wednesday, Thursday, Friday, ...    Sunday      07:00   
4                                           Flexible  Flexible      14:00   

  ShiftEnd  ContractHoursPerWeek  MaxDailyHours  MaxWeeklyHours     Status  
0    13:00                  

In [3]:
#testing data validations
from validator import validate_carers, validate_visits

carers = validate_carers(carers)
visits = validate_visits(visits)

print(carers.shape)
print(visits.shape)

[VALIDATOR] Checking carers data...
[VALIDATOR] Carers cleaned: (263, 12) → (263, 12)
[VALIDATOR] Checking visits data...
[VALIDATOR] Visits cleaned: (657, 9) → (657, 9)
(263, 12)
(657, 9)


In [4]:
#testing Preprocessing
from preprocessing import preprocess_carers, preprocess_visits

carers = preprocess_carers(carers)
visits = preprocess_visits(visits)

display(carers.head())
display(visits.head())

[PREPROCESS] Carers processing...
[PREPROCESS] Carers done
[PREPROCESS] Visits processing...
[PREPROCESS] Visits done


,CarerID,Latitude,Longitude,MaxTravelDistanceMiles,WorkingDays,OffDays,ShiftStart,ShiftEnd,ContractHoursPerWeek,MaxDailyHours,MaxWeeklyHours,Status,ShiftStartMin,ShiftEndMin,StatusBinary
0,C0001,51.372966,-0.357305,5,"Monday, Tuesday, Wednesday, Thursday, Friday, ...",Sunday,07:00,13:00,20,5,20,Available,420,780,1
1,C0001,51.372966,-0.357305,5,"Monday, Tuesday, Wednesday, Thursday, Friday, ...",Sunday,07:00,13:00,20,5,20,Available,420,780,1
2,C0001,51.372966,-0.357305,5,"Monday, Tuesday, Wednesday, Thursday, Friday, ...",Sunday,07:00,13:00,20,5,20,Available,420,780,1
3,C0001,51.372966,-0.357305,5,"Monday, Tuesday, Wednesday, Thursday, Friday, ...",Sunday,07:00,13:00,20,5,20,Available,420,780,1
4,C0002,51.310247,-0.362365,8,Flexible,Flexible,14:00,18:00,40,9,45,Available,840,1080,1


,VisitID,PatientID,Day,Latitude,Longitude,VisitDurationMinutes,TimeWindowStart,TimeWindowEnd,PriorityLevel,TimeWindowStartMin,TimeWindowEndMin,DayEncoded
0,V000001,P0001,Tuesday,51.360080,-0.273553,45,15:00,18:00,Low,900,1080,1
1,V000009,P0003,Friday,51.599749,0.100166,60,17:00,20:00,Medium,1020,1200,4
2,V000010,P0003,Thursday,51.599749,0.100166,60,17:00,20:00,Medium,1020,1200,3
3,V000011,P0003,Tuesday,51.599749,0.100166,60,17:00,20:00,Medium,1020,1200,1
4,V000012,P0003,Wednesday,51.599749,0.100166,60,17:00,20:00,Medium,1020,1200,2


In [8]:
import sys
import importlib

sys.path.append("/content/project/modules")

import config
importlib.reload(config)

print(config.EARTH_RADIUS_MILES)

3958.8


In [6]:
with open('/content/RB_Project/modules/config.py', 'r') as f:
    print(f.read())

DAY_MAP = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}


In [9]:
#testing Distance Engine
from distance_engine import build_distance_matrix

distance_matrix = build_distance_matrix(visits, carers)

print(distance_matrix.shape)
print(distance_matrix[:2, :2])

[DISTANCE ENGINE] Building distance matrix...
[DISTANCE ENGINE] Done. Shape: (657, 263)
(657, 263)
[[ 3.7209783  3.7209783]
 [25.158148  25.158148 ]]


In [10]:
#testing constraint engine
from constraint_engine import prepare_carers, get_feasible_carers

carers = prepare_carers(carers)

test_visit = visits.iloc[0]

feasible = get_feasible_carers(carers, test_visit)

print("Feasible carers:", len(feasible))
print(feasible[:10])

Feasible carers: 59
[17, 18, 19, 20, 21, 57, 58, 59, 60, 61]


In [12]:
#testing scheduler rule based
from scheduler_rule_based import run_rule_based_scheduler

schedule_df, unassigned = run_rule_based_scheduler(
    visits,
    carers,
    distance_matrix
)

print(schedule_df.head())
print("Unassigned visits:", len(unassigned))

[SCHEDULER] Running rule-based scheduling...
[SCHEDULER] Done
   VisitID CarerID  Distance  VisitDuration
0  V000001   C0039  2.206059             45
1  V000009   C0055  5.860516             60
2  V000010   C0055  5.860516             60
3  V000011   C0055  5.860516             60
4  V000012   C0055  5.860516             60
Unassigned visits: 102


In [13]:
from metrics import *

print("===== EVALUATION RESULTS =====")

coverage = coverage_rate(schedule_df, len(visits))
print("Coverage:", coverage)

print("Unassigned:", len(unassigned))

print("Total Distance:", total_distance(schedule_df))
print("Avg Distance:", average_distance(schedule_df))

workload = compute_workload(schedule_df)

print("Workload Variance:", workload_variance(workload))
print("Workload Std:", workload_std(workload))
print("Fairness (Jain):", jain_fairness(workload))

===== EVALUATION RESULTS =====
Coverage: 0.8447488584474886
Unassigned: 102
Total Distance: 3237.6372
Avg Distance: 5.8335805
Workload Variance: 494200.0
Workload Std: 702.993598832877
Fairness (Jain): 0.8199110851978719


In [16]:
import importlib
import metrics # Import the module explicitly
import constraint_engine # Import the module explicitly

importlib.reload(metrics) # Reload metrics to ensure latest changes are picked up
importlib.reload(constraint_engine) # Reload constraint_engine as well, for consistency if it changes

from metrics import *
from constraint_engine import get_feasible_carers

In [17]:
feasibility_rate, infeasible = feasibility_analysis(
    visits,
    carers,
    get_feasible_carers
)

print("Feasibility Rate:", feasibility_rate)
print("Sample Infeasible:", infeasible[:10])

Feasibility Rate: 0.8858447488584474
Sample Infeasible: ['V000051', 'V000052', 'V000095', 'V000102', 'V000111', 'V000112', 'V000344', 'V000425', 'V000426', 'V000435']


In [18]:
report = bottleneck_diagnosis(
    visits,
    carers,
    schedule_df,
    get_feasible_carers
)

print(report)

{'feasibility_rate': 0.8858447488584474, 'bottleneck': 'no_day_match', 'breakdown': {'no_available_carers': 0.0, 'no_day_match': 1.0, 'no_time_match': 0.0, 'no_workload_capacity': 0.0}, 'infeasible_visits_sample': ['V000051', 'V000052', 'V000095', 'V000102', 'V000111', 'V000112', 'V000344', 'V000425', 'V000426', 'V000435']}


In [19]:
from metrics import *

print("===== EVALUATION RESULTS =====")

coverage = coverage_rate(schedule_df, len(visits))
print("Coverage:", coverage)

print("Unassigned:", len(unassigned))

print("Total Distance:", total_distance(schedule_df))
print("Avg Distance:", average_distance(schedule_df))

workload = compute_workload(schedule_df)

print("Workload Variance:", workload_variance(workload))
print("Workload Std:", workload_std(workload))
print("Fairness (Jain):", jain_fairness(workload))

===== EVALUATION RESULTS =====
Coverage: 0.8447488584474886
Unassigned: 102
Total Distance: 3237.6372
Avg Distance: 5.8335805
Workload Variance: 494200.0
Workload Std: 702.993598832877
Fairness (Jain): 0.8199110851978719


In [23]:
import importlib
import metrics
importlib.reload(metrics)

<module 'metrics' from '/content/RB_Project/modules/metrics.py'>

In [24]:
from metrics import export_metrics_to_csv

In [27]:
import os

os.makedirs("results", exist_ok=True)
print(os.listdir("results"))

[]


In [28]:
export_metrics_to_csv(
    output_path="/content/rule_based_metrics.csv",
    coverage=coverage,
    unassigned_count=len(unassigned),
    total_distance=total_distance(schedule_df),
    avg_distance=average_distance(schedule_df),
    workload_variance=workload_variance(workload),
    workload_std=workload_std(workload),
    fairness=jain_fairness(workload),
    feasibility_rate=feasibility_rate,
    bottleneck=report["bottleneck"]
)

[METRICS] Saved to /content/rule_based_metrics.csv
